# 🚀 z/OS MIPS Prediction - Complete All-in-One Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chelvy/Perf_Plan/blob/main/notebooks/01_complete_pipeline.ipynb)

**Système complet de prédiction MIPS pour z/OS avec Machine Learning**

---

## 📋 Ce Notebook Inclut:

✅ **Transformation de données pivot** → Format ML  
✅ **Upload de vos propres données** → Validation automatique  
✅ **Création de données d'exemple** → Pour tester  
✅ **Entraînement complet** → Régression + Classification  
✅ **Évaluation et visualisation** → Comparaison de modèles  
✅ **Téléchargement des modèles** → Prêt pour production  

---

## ⏱️ Temps d'Exécution:

- **Avec données d'exemple:** 5-8 minutes  
- **Avec transformation pivot:** 8-12 minutes  
- **Avec vos données:** 10-15 minutes (selon taille)  

---

## 🎯 Quick Start:

**Pour données d'exemple (test):**  
→ Exécutez toutes les cellules (`Runtime` → `Run all`)  

**Pour vos données pivot:**  
→ Allez à **Section 3A** et uploadez votre fichier  

**Pour vos données long-format:**  
→ Allez à **Section 3B** et uploadez votre fichier  

---

---
# 📦 SECTION 1: Setup Environment

Installation automatique de toutes les dépendances.

In [ ]:
# Check environment
try:
    import google.colab
    IN_COLAB = True
    print("✅ Running on Google Colab")
except:
    IN_COLAB = False
    print("⚠️  Not running on Colab - some features may not work")

import os
import sys
from pathlib import Path

print(f"\n📍 Current directory: {os.getcwd()}")

In [ ]:
%%bash
# Clone repository
if [ -d "Perf_Plan" ]; then
    echo "📦 Repository exists, pulling latest..."
    cd Perf_Plan && git pull
else
    echo "📥 Cloning repository..."
    git clone https://github.com/chelvy/Perf_Plan.git
fi

echo "✅ Repository ready"

In [ ]:
# Change to project directory
if IN_COLAB:
    os.chdir('/content/Perf_Plan')
else:
    if Path('Perf_Plan').exists():
        os.chdir('Perf_Plan')

print(f"✅ Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt

print("\n✅ All dependencies installed!")

In [ ]:
# Import all required modules
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Add src to path
sys.path.insert(0, 'src')

# Import project modules
from src.data_loader import MIPSDataLoader, create_sample_data
from src.preprocessing import MIPSPreprocessor, create_mips_categories
from src.features import MIPSFeatureEngineering
from src.models.regression import RegressionModelFactory, BaselinePredictor
from src.models.classification import ClassificationModelFactory, MajorityClassBaseline
from src.training import MIPSTrainingPipeline
from src.evaluation import ModelEvaluator, ErrorAnalyzer
from src.visualization import MIPSVisualizer

# Settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
%matplotlib inline

print("✅ All modules imported successfully!")
print("\n" + "="*80)
print("🎯 READY TO START!")
print("="*80)

---
# 📊 SECTION 2: Configuration

**⚙️ Configurez votre pipeline ici**

In [ ]:
# ============================================================================
# CONFIGURATION - MODIFIEZ ICI
# ============================================================================

# 🔧 DATA SOURCE CONFIGURATION
# Choisissez UNE SEULE option en mettant True:

USE_PIVOT_DATA = False      # ⬅️ Pour données pivot (dates en colonnes)
USE_UPLOADED_DATA = False   # ⬅️ Pour données long-format (CSV standard)
USE_SAMPLE_DATA = True      # ⬅️ Pour données d'exemple (test)

# 🎯 TRAINING CONFIGURATION
CONFIG = {
    'test_size': 0.2,              # 20% pour test
    'random_state': 42,            # Reproductibilité
    'time_based_split': False,     # False = random split, True = time-based
    'scaler_type': 'standard',     # 'standard', 'minmax', or 'robust'
    'handle_outliers': True,       # Gérer les outliers
    'outlier_method': 'clip',      # 'clip' or 'remove'
    'feature_engineering': True,   # Activer feature engineering
    'n_categories': 3,             # 3 = LOW/MEDIUM/HIGH
    'category_method': 'quantile', # 'quantile' or 'equal'
    'save_models': True,           # Sauvegarder les modèles
    'output_dir': 'models',       # Répertoire de sortie
    'results_dir': 'results'      # Répertoire des résultats
}

# 📊 SAMPLE DATA CONFIGURATION (si USE_SAMPLE_DATA = True)
SAMPLE_CONFIG = {
    'n_samples': 2000,    # Nombre de records
    'n_apps': 100,        # Nombre d'applications
    'random_state': 42
}

# ============================================================================

print("⚙️  Configuration loaded:")
print(f"\n   Data Source:")
print(f"   - Pivot data: {USE_PIVOT_DATA}")
print(f"   - Uploaded data: {USE_UPLOADED_DATA}")
print(f"   - Sample data: {USE_SAMPLE_DATA}")
print(f"\n   Training:")
for key, value in CONFIG.items():
    print(f"   - {key}: {value}")

# Validation
sources_active = sum([USE_PIVOT_DATA, USE_UPLOADED_DATA, USE_SAMPLE_DATA])
if sources_active == 0:
    print("\n⚠️  WARNING: No data source selected! Set one to True.")
elif sources_active > 1:
    print("\n⚠️  WARNING: Multiple data sources selected! Choose only ONE.")
else:
    print("\n✅ Configuration valid!")

---
# 📤 SECTION 3: Data Preparation

Choisissez UNE option selon votre type de données.

## 3A. 🔄 Transform Pivot Data (Option 1)

**Utilisez cette section si vos données ont:**
- Des **dates en colonnes** (2022-01-01, 2022-02-01, ...)
- Des **indicateurs en lignes** (M24H, MDIU, MPTE, ...)
- Des **virgules** comme séparateurs décimaux

**⚠️ N'exécutez cette section QUE si `USE_PIVOT_DATA = True`**

In [ ]:
if USE_PIVOT_DATA and IN_COLAB:
    from google.colab import files
    
    print("="*80)
    print("🔄 UPLOAD PIVOT DATA")
    print("="*80)
    print("\n📋 Format attendu:")
    print("   code_application | code_indicateur | 2022-01-01 | 2022-02-01 | ...")
    print("   APP-001         | M24H           | 1234,5     | 1456,8     | ...")
    print("   APP-001         | MDIU           | 0,27       | 0,20       | ...")
    print("\n📊 Formats acceptés: CSV, TSV, Excel (.xlsx)")
    print("\n" + "="*80 + "\n")
    
    # Upload
    uploaded = files.upload()
    
    if uploaded:
        PIVOT_FILE = list(uploaded.keys())[0]
        print(f"\n✅ File uploaded: {PIVOT_FILE}")
        print(f"   Size: {len(uploaded[PIVOT_FILE]) / 1024:.2f} KB")
    else:
        print("\n⚠️  No file uploaded")
        PIVOT_FILE = None
        
elif USE_PIVOT_DATA:
    print("⚠️  Not on Colab. Set PIVOT_FILE path manually.")
    PIVOT_FILE = None
else:
    print("ℹ️  Pivot data not selected (USE_PIVOT_DATA = False)")
    PIVOT_FILE = None

In [ ]:
# Transform pivot data
if USE_PIVOT_DATA and PIVOT_FILE:
    print("🔄 Transforming pivot data to long format...\n")
    
    # Use transformer module
    from src.transform_pivot import PivotTransformer
    
    transformer = PivotTransformer()
    df_transformed, output_path = transformer.transform_file(
        input_path=PIVOT_FILE,
        output_path='data/transformed_data.csv'
    )
    
    print(f"\n✅ Transformation complete!")
    print(f"   Output: {output_path}")
    print(f"   Shape: {df_transformed.shape}")
    print(f"   Columns: {list(df_transformed.columns)}")
    
    # Preview
    print("\n🔍 Preview:")
    display(df_transformed.head())
    
    # Set DATA_PATH
    DATA_PATH = output_path
    print(f"\n✅ DATA_PATH set to: {DATA_PATH}")
    
elif USE_PIVOT_DATA:
    print("⚠️  Pivot file not loaded. Upload file first.")
else:
    print("ℹ️  Skipping pivot transformation (not selected)")

## 3B. 📤 Upload Long-Format Data (Option 2)

**Utilisez cette section si vos données sont déjà au format long:**
- Une ligne par observation
- Colonnes: application, timestamp, M24H, MDIU, MPTE, TXDIU, EFF, TVDIU, MIPS_consumption

**⚠️ N'exécutez cette section QUE si `USE_UPLOADED_DATA = True`**

In [ ]:
if USE_UPLOADED_DATA and IN_COLAB:
    from google.colab import files
    
    # Download template first
    print("📥 STEP 1: Download CSV Template (Optional)\n")
    
    template_data = {
        'application': ['APP_001', 'APP_002', 'APP_003'],
        'timestamp': ['2024-01-01', '2024-01-01', '2024-01-01'],
        'M24H': [1000.0, 1500.0, 900.0],
        'MDIU': [800.0, 1200.0, 750.0],
        'MPTE': [1200.0, 1800.0, 1100.0],
        'TXDIU': [50.0, 75.0, 45.0],
        'EFF': [0.95, 0.90, 0.92],
        'TVDIU': [100.0, 150.0, 90.0],
        'MIPS_consumption': [950.0, 1350.0, 850.0]
    }
    
    template_df = pd.DataFrame(template_data)
    template_df.to_csv('mips_data_template.csv', index=False)
    
    print("📋 Template format:")
    display(template_df)
    
    files.download('mips_data_template.csv')
    print("\n✅ Template downloaded!\n")
    print("="*80)
    
elif USE_UPLOADED_DATA:
    print("ℹ️  Template download only works on Colab")

In [ ]:
if USE_UPLOADED_DATA and IN_COLAB:
    from google.colab import files
    
    print("="*80)
    print("📤 UPLOAD YOUR DATA")
    print("="*80)
    print("\n📋 Required columns:")
    print("   application, timestamp, M24H, MDIU, MPTE,")
    print("   TXDIU, EFF, TVDIU, MIPS_consumption")
    print("\n📊 Recommended: 1000+ records")
    print("\n" + "="*80 + "\n")
    
    # Upload
    uploaded = files.upload()
    
    if uploaded:
        uploaded_filename = list(uploaded.keys())[0]
        DATA_PATH = uploaded_filename
        
        print(f"\n✅ File uploaded: {uploaded_filename}")
        print(f"   Size: {len(uploaded[uploaded_filename]) / 1024:.2f} KB")
        
        # Quick validation
        try:
            data_check = pd.read_csv(DATA_PATH)
            print(f"   Records: {len(data_check):,}")
            print(f"   Columns: {len(data_check.columns)}")
            
            required_cols = ['application', 'M24H', 'MDIU', 'MPTE', 'TXDIU', 
                           'EFF', 'TVDIU', 'MIPS_consumption']
            missing = [col for col in required_cols if col not in data_check.columns]
            
            if missing:
                print(f"\n⚠️  WARNING: Missing columns: {missing}")
            else:
                print("\n✅ All required columns present!")
                
            print("\n🔍 Preview:")
            display(data_check.head())
            
        except Exception as e:
            print(f"\n⚠️  Error reading file: {e}")
    else:
        print("\n⚠️  No file uploaded")
        DATA_PATH = None
        
elif USE_UPLOADED_DATA:
    print("⚠️  Not on Colab. Set DATA_PATH manually.")
    DATA_PATH = None
else:
    print("ℹ️  Upload not selected (USE_UPLOADED_DATA = False)")

## 3C. 🎲 Create Sample Data (Option 3)

**Utilisez cette section pour tester le système avec des données synthétiques.**

**⚠️ N'exécutez cette section QUE si `USE_SAMPLE_DATA = True`**

In [ ]:
if USE_SAMPLE_DATA:
    print("🎲 Creating sample z/OS MIPS data...\n")
    
    sample_path = 'data/sample/sample_mips_data.csv'
    df_sample = create_sample_data(
        output_path=sample_path,
        n_samples=SAMPLE_CONFIG['n_samples'],
        n_apps=SAMPLE_CONFIG['n_apps'],
        random_state=SAMPLE_CONFIG['random_state']
    )
    
    print(f"\n✅ Sample data created!")
    print(f"   Records: {len(df_sample):,}")
    print(f"   Applications: {df_sample['application'].nunique()}")
    print(f"   Date range: {df_sample['timestamp'].min()} to {df_sample['timestamp'].max()}")
    
    print("\n🔍 Preview:")
    display(df_sample.head())
    
    # Set DATA_PATH
    DATA_PATH = sample_path
    print(f"\n✅ DATA_PATH set to: {DATA_PATH}")
    
else:
    print("ℹ️  Sample data not selected (USE_SAMPLE_DATA = False)")

## ✅ Data Preparation Complete

Verify that DATA_PATH is set correctly:

In [ ]:
# Verify DATA_PATH
if 'DATA_PATH' in locals() and DATA_PATH:
    print("="*80)
    print("✅ DATA READY")
    print("="*80)
    print(f"\nData file: {DATA_PATH}")
    
    if os.path.exists(DATA_PATH):
        data_check = pd.read_csv(DATA_PATH)
        print(f"Records: {len(data_check):,}")
        print(f"Columns: {list(data_check.columns)}")
        
        # Update CONFIG
        CONFIG['data_path'] = DATA_PATH
        print("\n✅ Configuration updated with data path")
    else:
        print("\n⚠️  WARNING: File does not exist!")
else:
    print("="*80)
    print("⚠️  ERROR: DATA_PATH NOT SET")
    print("="*80)
    print("\nGo back and:")
    print("1. Set ONE data source to True in Section 2")
    print("2. Execute the corresponding section in Section 3")
    print("="*80)

---
# 📊 SECTION 4: Data Exploration

Explorez vos données avant l'entraînement.

In [ ]:
# Load data
if 'DATA_PATH' in locals() and DATA_PATH:
    loader = MIPSDataLoader(DATA_PATH)
    data = loader.load_csv()
    
    print("="*80)
    print("📊 DATA SUMMARY")
    print("="*80)
    
    info = loader.get_data_info()
    for key, value in info.items():
        if key != 'missing_values':
            print(f"  {key}: {value}")
    
    print("\n📋 First 10 records:")
    display(data.head(10))
    
    print("\n📈 Statistical Summary:")
    display(data.describe())
else:
    print("⚠️  No data loaded. Complete Section 3 first.")

In [ ]:
# Visualizations
if 'data' in locals():
    # MIPS distribution
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Histogram
    axes[0].hist(data['MIPS_consumption'], bins=50, edgecolor='black', 
                 alpha=0.7, color='green')
    axes[0].set_title('MIPS Consumption Distribution', fontweight='bold', fontsize=14)
    axes[0].set_xlabel('MIPS')
    axes[0].set_ylabel('Frequency')
    axes[0].grid(True, alpha=0.3)
    
    # Box plot
    axes[1].boxplot(data['MIPS_consumption'])
    axes[1].set_title('MIPS Box Plot', fontweight='bold', fontsize=14)
    axes[1].set_ylabel('MIPS')
    axes[1].grid(True, alpha=0.3)
    
    # Log scale
    axes[2].hist(np.log1p(data['MIPS_consumption']), bins=50, 
                 edgecolor='black', alpha=0.7, color='blue')
    axes[2].set_title('MIPS Distribution (Log Scale)', fontweight='bold', fontsize=14)
    axes[2].set_xlabel('log(MIPS + 1)')
    axes[2].set_ylabel('Frequency')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print(f"\n📊 MIPS Statistics:")
    print(f"  Mean: {data['MIPS_consumption'].mean():.2f}")
    print(f"  Median: {data['MIPS_consumption'].median():.2f}")
    print(f"  Std: {data['MIPS_consumption'].std():.2f}")
    print(f"  Range: [{data['MIPS_consumption'].min():.2f}, {data['MIPS_consumption'].max():.2f}]")

In [ ]:
# Correlation matrix
if 'data' in locals():
    numeric_cols = data.select_dtypes(include=[np.number]).columns
    correlation_matrix = data[numeric_cols].corr()
    
    plt.figure(figsize=(12, 10))
    sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm',
                center=0, square=True, linewidths=1, cbar_kws={'shrink': 0.8})
    plt.title('Correlation Matrix - MIPS Indicators', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.show()

---
# 🚀 SECTION 5: Train Models

Entraînement complet de tous les modèles.

In [ ]:
# Create and run training pipeline
if 'DATA_PATH' in locals() and DATA_PATH:
    print("="*80)
    print("🎯 STARTING TRAINING PIPELINE")
    print("="*80)
    print(f"\nData: {DATA_PATH}")
    print(f"Mode: BOTH (Regression + Classification)")
    print(f"Feature Engineering: {CONFIG['feature_engineering']}")
    print("\n" + "="*80 + "\n")
    
    # Create pipeline
    pipeline = MIPSTrainingPipeline(CONFIG)
    
    # Run full pipeline
    results = pipeline.run_full_pipeline(mode='both')
    
    print("\n" + "="*80)
    print("✅ TRAINING COMPLETED!")
    print("="*80)
    
else:
    print("⚠️  Cannot train: DATA_PATH not set")

---
# 📈 SECTION 6: Regression Results

Analyse des résultats de régression.

In [ ]:
# Compare regression models
if 'results' in locals() and 'regression' in results:
    evaluator = ModelEvaluator()
    
    # RMSE comparison
    rmse_comparison = evaluator.compare_models(
        results['regression'],
        metric='rmse',
        mode='regression'
    )
    
    print("="*80)
    print("📊 REGRESSION MODELS - RMSE COMPARISON")
    print("="*80)
    display(rmse_comparison.head(15))
    
    # R² comparison  
    r2_comparison = evaluator.compare_models(
        results['regression'],
        metric='r2',
        mode='regression'
    ).sort_values('test_r2', ascending=False)
    
    print("\n" + "="*80)
    print("📊 REGRESSION MODELS - R² COMPARISON")
    print("="*80)
    display(r2_comparison.head(15))
    
else:
    print("⚠️  No regression results available")

In [ ]:
# Visualize regression comparison
if 'rmse_comparison' in locals():
    viz = MIPSVisualizer(output_dir='results')
    
    viz.plot_model_comparison(
        rmse_comparison.head(15),
        metric='rmse',
        title='Regression Models - RMSE Comparison'
    )

In [ ]:
# Analyze best regression model
if 'results' in locals() and 'regression' in results and hasattr(pipeline, 'best_model_name'):
    best_name = pipeline.best_model_name
    best_result = results['regression'][best_name]
    
    print("="*80)
    print(f"🏆 BEST REGRESSION MODEL: {best_name}")
    print("="*80)
    
    evaluator.print_regression_summary(
        best_result['test_metrics'],
        best_name
    )
    
    # Get predictions
    y_test = best_result['predictions']['test']
    
    # Get y_true
    loader_temp = MIPSDataLoader(DATA_PATH)
    data_temp = loader_temp.load_csv()
    train_temp, test_temp = loader_temp.split_train_test(
        test_size=CONFIG['test_size'],
        random_state=CONFIG['random_state']
    )
    _, y_true = loader_temp.get_feature_target_split(test_temp)
    
    # Visualizations
    viz.plot_predictions_vs_actual(
        y_true,
        y_test,
        title=f"{best_name} - Predictions vs Actual"
    )
    
    viz.plot_residuals(
        y_true,
        y_test,
        title=f"{best_name} - Residual Analysis"
    )

---
# 🎯 SECTION 7: Classification Results

Analyse des résultats de classification.

In [ ]:
# Compare classification models
if 'results' in locals() and 'classification' in results:
    evaluator = ModelEvaluator()
    
    # Accuracy comparison
    acc_comparison = evaluator.compare_models(
        results['classification'],
        metric='accuracy',
        mode='classification'
    )
    
    print("="*80)
    print("📊 CLASSIFICATION MODELS - ACCURACY COMPARISON")
    print("="*80)
    print("\nMetric: Accuracy = (1/n) × Σ 𝟙[ŷᵢ = yᵢ]\n")
    display(acc_comparison.head(15))
    
    # F1 comparison
    f1_comparison = evaluator.compare_models(
        results['classification'],
        metric='f1_macro',
        mode='classification'
    )
    
    print("\n" + "="*80)
    print("📊 CLASSIFICATION MODELS - F1 SCORE COMPARISON")
    print("="*80)
    display(f1_comparison.head(15))
    
else:
    print("⚠️  No classification results available")

In [ ]:
# Visualize classification comparison
if 'acc_comparison' in locals():
    viz = MIPSVisualizer(output_dir='results')
    
    viz.plot_model_comparison(
        acc_comparison.head(15),
        metric='accuracy',
        title='Classification Models - Accuracy Comparison'
    )

In [ ]:
# Analyze best classification model
if 'acc_comparison' in locals() and len(acc_comparison) > 0:
    best_clf_name = acc_comparison.iloc[0]['model']
    best_clf_result = results['classification'][best_clf_name]
    
    print("="*80)
    print(f"🏆 BEST CLASSIFICATION MODEL: {best_clf_name}")
    print("="*80)
    
    evaluator.print_classification_summary(
        best_clf_result['test_metrics'],
        best_clf_name
    )
    
    # Confusion matrix
    cm = np.array(best_clf_result['test_metrics']['confusion_matrix'])
    classes = ['LOW', 'MEDIUM', 'HIGH'][:CONFIG['n_categories']]
    
    viz.plot_confusion_matrix(
        cm,
        classes=classes,
        title=f"{best_clf_name} - Confusion Matrix",
        normalize=False
    )
    
    viz.plot_confusion_matrix(
        cm,
        classes=classes,
        title=f"{best_clf_name} - Normalized Confusion Matrix",
        normalize=True
    )

---
# 📋 SECTION 8: Summary & Final Comparison

In [ ]:
# Final summary
if 'results' in locals():
    print("="*80)
    print("📋 FINAL SUMMARY")
    print("="*80)
    
    if 'regression' in results:
        print("\n🔵 REGRESSION RESULTS:")
        print(f"   Best Model: {pipeline.best_model_name}")
        best_reg_metrics = results['regression'][pipeline.best_model_name]['test_metrics']
        print(f"   Test RMSE: {best_reg_metrics['rmse']:.2f}")
        print(f"   Test R²: {best_reg_metrics['r2']:.4f}")
        print(f"   Test MAE: {best_reg_metrics['mae']:.2f}")
        print(f"   Test MAPE: {best_reg_metrics['mape']:.2f}%")
        
        # Compare with baseline
        if 'baseline_mean' in results['regression']:
            baseline_rmse = results['regression']['baseline_mean']['test_metrics']['rmse']
            improvement = ((baseline_rmse - best_reg_metrics['rmse']) / baseline_rmse) * 100
            print(f"   \n   📈 Improvement over baseline: {improvement:.1f}%")
    
    if 'classification' in results:
        print("\n🟢 CLASSIFICATION RESULTS:")
        best_clf_name = acc_comparison.iloc[0]['model']
        print(f"   Best Model: {best_clf_name}")
        best_clf_metrics = results['classification'][best_clf_name]['test_metrics']
        print(f"   Test Accuracy: {best_clf_metrics['accuracy']:.4f}")
        print(f"   Test F1 (macro): {best_clf_metrics['f1_macro']:.4f}")
        print(f"   Test Precision (macro): {best_clf_metrics['precision_macro']:.4f}")
        print(f"   Test Recall (macro): {best_clf_metrics['recall_macro']:.4f}")
        
        # Compare with baseline
        if 'baseline_majority' in results['classification']:
            baseline_acc = results['classification']['baseline_majority']['test_metrics']['accuracy']
            improvement = ((best_clf_metrics['accuracy'] - baseline_acc) / baseline_acc) * 100
            print(f"   \n   📈 Improvement over baseline: {improvement:.1f}%")
    
    print("\n" + "="*80)
    print("\n✅ All models trained and evaluated successfully!")
    print("📁 Models saved to: models/")
    print("📁 Results saved to: results/")
    print("="*80)

---
# 💾 SECTION 9: Download Models & Results

In [ ]:
# List saved files
print("📦 Saved Models and Results:\n")

if os.path.exists('models'):
    models_files = list(Path('models').glob('*'))
    if models_files:
        print("Models directory:")
        for f in models_files:
            size_mb = f.stat().st_size / (1024 * 1024)
            print(f"  📄 {f.name} ({size_mb:.2f} MB)")

if os.path.exists('results'):
    results_files = list(Path('results').glob('*'))
    if results_files:
        print("\nResults directory:")
        for f in results_files:
            size_kb = f.stat().st_size / 1024
            print(f"  📄 {f.name} ({size_kb:.2f} KB)")

In [ ]:
# Download as ZIP (Colab only)
if IN_COLAB:
    from google.colab import files
    import zipfile
    
    print("📦 Creating ZIP archive...\n")
    
    zip_filename = 'zos_mips_models_results.zip'
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
        # Add models
        if os.path.exists('models'):
            for file in Path('models').rglob('*'):
                if file.is_file():
                    zipf.write(file, arcname=f'models/{file.name}')
        
        # Add results
        if os.path.exists('results'):
            for file in Path('results').rglob('*'):
                if file.is_file():
                    zipf.write(file, arcname=f'results/{file.name}')
    
    print(f"✅ ZIP created: {zip_filename}")
    print(f"   Size: {Path(zip_filename).stat().st_size / (1024*1024):.2f} MB\n")
    
    print("⬇️  Downloading...")
    files.download(zip_filename)
    print("✅ Download complete!")
else:
    print("ℹ️  Not on Colab - files saved locally")

---
# 🔮 SECTION 10: Make Predictions (Optional)

Utilisez le modèle entraîné pour faire des prédictions.

In [ ]:
# Create new sample data for prediction
if 'pipeline' in locals() and hasattr(pipeline, 'best_model'):
    print("🔮 Creating new data for prediction...\n")
    
    new_data = pd.DataFrame({
        'application': ['APP_001', 'APP_002', 'APP_003', 'APP_004', 'APP_005'],
        'timestamp': pd.date_range('2025-01-01', periods=5, freq='D'),
        'M24H': [1200, 1500, 900, 1800, 1100],
        'MDIU': [1000, 1300, 800, 1600, 950],
        'MPTE': [1400, 1700, 1000, 2000, 1250],
        'TXDIU': [60, 75, 45, 85, 55],
        'EFF': [0.92, 0.88, 0.95, 0.85, 0.90],
        'TVDIU': [120, 140, 100, 160, 110]
    })
    
    print("📋 New data:")
    display(new_data)
    
    # Apply feature engineering if enabled
    if CONFIG['feature_engineering']:
        feature_eng = MIPSFeatureEngineering()
        new_data_fe = feature_eng.create_all_features(new_data)
    else:
        new_data_fe = new_data.copy()
    
    # Preprocess
    X_new = new_data_fe.drop(columns=['timestamp'], errors='ignore')
    X_new_scaled = pipeline.preprocessor.transform(X_new)
    
    # Predict
    predictions = pipeline.best_model.predict(X_new_scaled)
    
    print(f"\n🔮 Predictions from {pipeline.best_model_name}:\n")
    results_df = new_data[['application', 'M24H', 'MDIU', 'MPTE']].copy()
    results_df['Predicted_MIPS'] = predictions
    results_df['Predicted_MIPS'] = results_df['Predicted_MIPS'].round(2)
    
    display(results_df)
else:
    print("⚠️  No trained model available")

---
# 🎓 SECTION 11: Next Steps & Production Usage

## 📝 Utilisation des Modèles en Production

```python
# Charger le modèle entraîné
from src.models.regression import MIPSRegressionModel
from src.preprocessing import MIPSPreprocessor

model = MIPSRegressionModel.load('models/best_model.pkl')
preprocessor = MIPSPreprocessor.load('models/preprocessor.pkl')

# Préparer nouvelles données
X_new_scaled = preprocessor.transform(new_data)

# Prédire
predictions = model.predict(X_new_scaled)
```

## 🚀 Prochaines Étapes

1. **Uploadez vos vraies données z/OS** (3 ans d'historique recommandés)
2. **Fine-tuning des hyperparamètres** pour améliorer performance
3. **Déployez les modèles** dans votre environnement de production
4. **Monitoring continu** des performances
5. **Réentraînement périodique** avec nouvelles données

## 📚 Documentation

- **Guide complet:** [README.md](https://github.com/chelvy/Perf_Plan/blob/main/README.md)
- **Transform pivot:** [TRANSFORM_GUIDE.md](https://github.com/chelvy/Perf_Plan/blob/main/TRANSFORM_GUIDE.md)
- **Upload données:** [UPLOAD_GUIDE.md](https://github.com/chelvy/Perf_Plan/blob/main/UPLOAD_GUIDE.md)
- **CLI usage:** `python main.py --help`

## ✅ Checklist de Production

- [ ] Données de qualité (>1000 records, 3 ans)
- [ ] Validation croisée des modèles
- [ ] Tests sur données de production
- [ ] Monitoring des prédictions
- [ ] Plan de réentraînement (mensuel/trimestriel)
- [ ] Documentation du déploiement
- [ ] Backup des modèles

---

**🎉 Félicitations! Votre système de prédiction MIPS z/OS est opérationnel!**

---